In [ ]:
# Install required packages (uncomment for Colab)
!pip install -q datasets transformers peft accelerate scikit-learn

In [ ]:
# =============================================================================
# MOUNT GOOGLE DRIVE (if using Drive to store models)
# =============================================================================
# Uncomment if you uploaded models to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/trained_models /content/trained_models

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================
import os
import time
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import AutoModel, AutoTokenizer
from peft import LoraConfig, get_peft_model
from sklearn.metrics import precision_recall_fscore_support, f1_score
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

In [ ]:
# =============================================================================
# LOAD ENSEMBLE CONFIG
# =============================================================================

with open('trained_models/ensemble_config.pkl', 'rb') as f:
    ensemble_config = pickle.load(f)

# Extract config
weights = ensemble_config['weights']
thresholds = ensemble_config['thresholds']
all_labels = ensemble_config['all_labels']
label2idx = ensemble_config['label2idx']
idx2label = ensemble_config['idx2label']
CONFIG = ensemble_config['config']
MODEL_CONFIGS = ensemble_config['model_configs']
LANG_LABELS = ensemble_config['lang_labels']
LANGUAGES = ensemble_config['languages']
NUM_LABELS = ensemble_config['num_labels']
val_f1_scores = ensemble_config['val_f1_scores']

print("Ensemble config loaded!")
print(f"\nValidation F1 scores from training:")
for k, v in val_f1_scores.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
# =============================================================================
# MODEL DEFINITION (same as training)
# =============================================================================

class XLoRAClassifier(nn.Module):
    def __init__(self, model_name, hidden_size, num_labels, lora_config):
        super().__init__()
        self.base_model = AutoModel.from_pretrained(model_name)
        self.base_model = get_peft_model(self.base_model, lora_config)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state
        mask = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
        pooled = (hidden_states * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        return self.classifier(self.dropout(pooled))

def load_model(model_key, cfg, num_labels):
    """Load model with weights."""
    lora_config = LoraConfig(
        r=CONFIG['lora_r'],
        lora_alpha=CONFIG['lora_alpha'],
        target_modules=cfg['target_modules'],
        lora_dropout=CONFIG['lora_dropout'],
        bias='none',
        task_type='FEATURE_EXTRACTION',
    )
    model = XLoRAClassifier(cfg['name'], cfg['hidden_size'], num_labels, lora_config)
    model.load_state_dict(torch.load(f'trained_models/{model_key}_model.pt', map_location=device))
    return model.to(device).eval()

In [ ]:
# =============================================================================
# LOAD ALL MODELS AND TOKENIZERS
# =============================================================================

print("Loading models...")
trained_models = {}
tokenizers = {}

for model_key, cfg in MODEL_CONFIGS.items():
    print(f"  Loading {model_key}...")
    trained_models[model_key] = load_model(model_key, cfg, NUM_LABELS)
    tokenizers[model_key] = AutoTokenizer.from_pretrained(cfg['name'])
    if tokenizers[model_key].pad_token is None:
        tokenizers[model_key].pad_token = tokenizers[model_key].eos_token

print("\nAll models loaded!")

In [ ]:
# =============================================================================
# LOAD TEST DATA FROM HUGGINGFACE
# =============================================================================

print('Loading test data from HuggingFace...')
ds = load_dataset('NLBSE/nlbse26-code-comment-classification')

def labels_to_unified(labels_list, lang):
    lang_labels = LANG_LABELS[lang]
    unified = [0] * NUM_LABELS
    for i, val in enumerate(labels_list):
        if val == 1:
            label_name = lang_labels[i]
            unified[label2idx[label_name]] = 1
    return unified

def prepare_data(split_name, lang):
    data = ds[split_name]
    texts = data['combo']
    labels = [labels_to_unified(l, lang) for l in data['labels']]
    return texts, labels

test_texts, test_labels, test_langs = [], [], []
for lang in LANGUAGES:
    texts, labels = prepare_data(f'{lang}_test', lang)
    test_texts.extend(texts)
    test_labels.extend(labels)
    test_langs.extend([lang] * len(texts))

print(f'Total test samples: {len(test_texts)}')

In [ ]:
# =============================================================================
# DATASET CLASS
# =============================================================================

class CommentDataset(Dataset):
    def __init__(self, texts, labels, languages, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.languages = languages
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.float),
            'language': self.languages[idx],
        }

In [ ]:
# =============================================================================
# GET TEST PREDICTIONS
# =============================================================================

def get_predictions(model, tokenizer, texts, labels, languages, batch_size=32):
    model.eval()
    ds = CommentDataset(texts, labels, languages, tokenizer, CONFIG['max_length'])
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
    
    all_probs, all_labels, all_langs = [], [], []
    with torch.no_grad():
        for batch in loader:
            logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(batch['labels'].numpy())
            all_langs.extend(batch['language'])
    
    return np.array(all_probs), np.array(all_labels), all_langs

print("Getting test predictions from all models...")
test_probs_all = []

for model_key in MODEL_CONFIGS.keys():
    print(f"  {model_key}...")
    model = trained_models[model_key]
    tokenizer = tokenizers[model_key]
    tp, tl, tlng = get_predictions(model, tokenizer, test_texts, test_labels, test_langs)
    test_probs_all.append(tp)

test_labels_arr = tl
test_langs_list = tlng
print("Done!")

In [ ]:
# =============================================================================
# ENSEMBLE PREDICTION
# =============================================================================

def ensemble_predict(model_probs, weights):
    num_samples, num_labels = model_probs[0].shape
    ensemble_probs = np.zeros((num_samples, num_labels))
    for label_idx in range(num_labels):
        for m, probs in enumerate(model_probs):
            ensemble_probs[:, label_idx] += weights[label_idx, m] * probs[:, label_idx]
    return ensemble_probs

def apply_thresholds(probs, thresholds, languages):
    preds = np.zeros_like(probs, dtype=int)
    for i, lang in enumerate(languages):
        preds[i] = (probs[i] >= thresholds.get(lang, np.ones(probs.shape[1]) * 0.5)).astype(int)
    return preds

# Compute ensemble probabilities
test_ens_probs = ensemble_predict(test_probs_all, weights)

# Apply thresholds
test_preds = apply_thresholds(test_ens_probs, thresholds, test_langs_list)

In [ ]:
# =============================================================================
# EVALUATE PER LANGUAGE/CATEGORY
# =============================================================================

results = []
for lang in LANGUAGES:
    mask = np.array([l == lang for l in test_langs_list])
    lp, lt = test_preds[mask], test_labels_arr[mask]
    
    for idx, name in enumerate(all_labels):
        y_true, y_pred = lt[:, idx], lp[:, idx]
        if y_true.sum() == 0:
            continue
        p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)
        results.append({'lan': lang, 'cat': name, 'precision': p, 'recall': r, 'f1': f1})

results_df = pd.DataFrame(results)

# Sort like baseline
baseline_order = [
    ('java', 'summary'), ('java', 'Ownership'), ('java', 'Expand'), ('java', 'usage'),
    ('java', 'Pointer'), ('java', 'deprecation'), ('java', 'rational'),
    ('python', 'Usage'), ('python', 'Parameters'), ('python', 'DevelopmentNotes'),
    ('python', 'Expand'), ('python', 'Summary'),
    ('pharo', 'Keyimplementationpoints'), ('pharo', 'Example'), ('pharo', 'Responsibilities'),
    ('pharo', 'Intent'), ('pharo', 'Keymessages'), ('pharo', 'Collaborators')
]

rows = []
for lan, cat in baseline_order:
    row = results_df[(results_df['lan'] == lan) & (results_df['cat'] == cat)]
    if len(row) > 0:
        rows.append(row.iloc[0].to_dict())
    else:
        rows.append({'lan': lan, 'cat': cat, 'precision': 0, 'recall': 0, 'f1': 0})

final_df = pd.DataFrame(rows)

print("\n" + "="*70)
print("FINAL RESULTS")
print("="*70)
print(final_df.to_string(index=False))

In [ ]:
# =============================================================================
# COMPUTE F1 METRICS
# =============================================================================

# Baseline results
baseline_data = {
    'java': {'summary': 0.8789, 'Ownership': 1.0, 'Expand': 0.3736, 'usage': 0.8670, 
             'Pointer': 0.8612, 'deprecation': 0.7778, 'rational': 0.3556},
    'python': {'Usage': 0.6739, 'Parameters': 0.7191, 'DevelopmentNotes': 0.3048, 
               'Expand': 0.5397, 'Summary': 0.6723},
    'pharo': {'Keyimplementationpoints': 0.6, 'Example': 0.8814, 'Responsibilities': 0.6813,
              'Intent': 0.7826, 'Keymessages': 0.5789, 'Collaborators': 0.1667}
}

baseline_f1_list = []
for lang in baseline_data:
    for cat in baseline_data[lang]:
        baseline_f1_list.append(baseline_data[lang][cat])

baseline_macro_f1 = np.mean(baseline_f1_list)
our_macro_f1 = final_df['f1'].mean()

# Compute support for weighted F1
support = {}
for lang in LANGUAGES:
    test_data = ds[f'{lang}_test']
    lang_labels = LANG_LABELS[lang]
    for labels in test_data['labels']:
        for i, val in enumerate(labels):
            if val == 1:
                key = (lang, lang_labels[i])
                support[key] = support.get(key, 0) + 1

final_df['support'] = final_df.apply(lambda r: support.get((r['lan'], r['cat']), 0), axis=1)
total_support = final_df['support'].sum()
our_weighted_f1 = (final_df['f1'] * final_df['support']).sum() / total_support

baseline_weighted_f1 = 0
for lang in baseline_data:
    for cat, f1 in baseline_data[lang].items():
        s = support.get((lang, cat), 0)
        baseline_weighted_f1 += f1 * s
baseline_weighted_f1 /= total_support

print("\n" + "="*70)
print("F1 SCORE COMPARISON")
print("="*70)
print(f"{'Metric':<20} {'Our Model':>15} {'Baseline':>15} {'Δ':>15}")
print("-"*70)
print(f"{'F1 Macro':<20} {our_macro_f1:>15.4f} {baseline_macro_f1:>15.4f} {our_macro_f1 - baseline_macro_f1:>+15.4f}")
print(f"{'F1 Weighted':<20} {our_weighted_f1:>15.4f} {baseline_weighted_f1:>15.4f} {our_weighted_f1 - baseline_weighted_f1:>+15.4f}")

In [ ]:
# =============================================================================
# RUNTIME MEASUREMENT (COMPETITION STANDARD - 10 RUNS)
# =============================================================================

print("\n" + "="*70)
print("RUNTIME MEASUREMENT (Competition Standard)")
print("="*70)

# Use 100 samples for runtime measurement
sample_texts = test_texts[:100]
sample_labels = test_labels[:100]
sample_langs = test_langs[:100]

# Warmup
print("Warming up...")
for _ in range(3):
    for model_key, model in trained_models.items():
        tok = tokenizers[model_key]
        enc = tok(sample_texts[0], truncation=True, max_length=CONFIG['max_length'], 
                  padding='max_length', return_tensors='pt')
        with torch.no_grad():
            model(enc['input_ids'].to(device), enc['attention_mask'].to(device))

if torch.cuda.is_available():
    torch.cuda.synchronize()

# Measure runtime - 10 runs as per competition
print("Measuring runtime (10 runs)...")
run_times = []

for run in range(10):
    t_start = time.time()
    
    for text in sample_texts:
        for model_key, model in trained_models.items():
            tok = tokenizers[model_key]
            enc = tok(text, truncation=True, max_length=CONFIG['max_length'],
                      padding='max_length', return_tensors='pt')
            with torch.no_grad():
                model(enc['input_ids'].to(device), enc['attention_mask'].to(device))
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    run_time = (time.time() - t_start) / len(sample_texts)
    run_times.append(run_time)
    print(f"  Run {run+1}: {run_time:.4f} s/sample")

avg_runtime = float(np.mean(run_times))
std_runtime = float(np.std(run_times))

print(f"\nAverage runtime: {avg_runtime:.4f} ± {std_runtime:.4f} s/sample")

In [ ]:
# =============================================================================
# GFLOPS ESTIMATION
# =============================================================================

# Count total parameters
total_params = sum(sum(p.numel() for p in m.parameters()) for m in trained_models.values())

# Estimate GFLOPS: 2 * params * seq_length (forward pass FLOPs)
gflops = (2 * total_params * CONFIG['max_length']) / 1e9

print(f"\nTotal parameters: {total_params:,}")
print(f"Estimated GFLOPS: {gflops:.2f}")

In [ ]:
# =============================================================================
# COMPUTE FINAL SUBMISSION SCORE
# =============================================================================

MAX_AVG_RUNTIME = 5.0  # seconds
MAX_AVG_GFLOPS = 5000.0

def compute_submission_score(avg_f1, avg_runtime, gflops):
    rt_comp = max((MAX_AVG_RUNTIME - avg_runtime) / MAX_AVG_RUNTIME, 0)
    gf_comp = max((MAX_AVG_GFLOPS - gflops) / MAX_AVG_GFLOPS, 0)
    score = 0.60 * avg_f1 + 0.20 * rt_comp + 0.20 * gf_comp
    return score, 0.60 * avg_f1, 0.20 * rt_comp, 0.20 * gf_comp

# Our submission score
our_score, our_f1_comp, our_rt_comp, our_gf_comp = compute_submission_score(
    our_macro_f1, avg_runtime, gflops
)

# Baseline submission score
baseline_runtime = 0.015
baseline_gflops = 15.0
baseline_score, base_f1_comp, base_rt_comp, base_gf_comp = compute_submission_score(
    baseline_macro_f1, baseline_runtime, baseline_gflops
)

print("\n" + "="*70)
print("FINAL SUBMISSION SCORE (Google Colab T4)")
print("="*70)
print(f"\n{'Component':<25} {'Our Model':>15} {'Baseline':>15}")
print("-"*70)
print(f"{'Avg F1':<25} {our_macro_f1:>15.4f} {baseline_macro_f1:>15.4f}")
print(f"{'Runtime (s/sample)':<25} {avg_runtime:>15.4f} {baseline_runtime:>15.4f}")
print(f"{'GFLOPS':<25} {gflops:>15.2f} {baseline_gflops:>15.2f}")
print("-"*70)
print(f"{'F1 Component (60%)':<25} {our_f1_comp:>15.4f} {base_f1_comp:>15.4f}")
print(f"{'Runtime Component (20%)':<25} {our_rt_comp:>15.4f} {base_rt_comp:>15.4f}")
print(f"{'GFLOPS Component (20%)':<25} {our_gf_comp:>15.4f} {base_gf_comp:>15.4f}")
print("="*70)
print(f"{'SUBMISSION SCORE':<25} {our_score:>15.4f} {baseline_score:>15.4f}")
print(f"{'Difference':<25} {our_score - baseline_score:>+15.4f}")
print("="*70)

if our_score > baseline_score:
    print("\n✓ OUR MODEL BEATS THE BASELINE!")
else:
    print(f"\n✗ Baseline ahead by {baseline_score - our_score:.4f}")

In [ ]:
# =============================================================================
# DETAILED COMPARISON TABLE
# =============================================================================

print("\n" + "="*70)
print("DETAILED COMPARISON WITH BASELINE")
print("="*70)

comparison = []
for _, row in final_df.iterrows():
    lang, cat = row['lan'], row['cat']
    our_f1 = row['f1']
    base_f1 = baseline_data.get(lang, {}).get(cat, 0)
    delta = our_f1 - base_f1
    comparison.append({
        'Language': lang,
        'Category': cat,
        'Baseline F1': f'{base_f1:.4f}',
        'Our F1': f'{our_f1:.4f}',
        'Δ F1': f'{delta:+.4f}',
        'Better': '✓' if delta > 0.001 else ('=' if abs(delta) < 0.001 else '✗')
    })

comp_df = pd.DataFrame(comparison)
print(comp_df.to_string(index=False))

num_better = sum(1 for c in comparison if c['Better'] == '✓')
num_worse = sum(1 for c in comparison if c['Better'] == '✗')
print(f"\nCategories improved: {num_better}/18")
print(f"Categories worse: {num_worse}/18")

In [ ]:
# =============================================================================
# SAVE RESULTS TO CSV
# =============================================================================

final_df[['lan', 'cat', 'precision', 'recall', 'f1']].to_csv('final_results.csv', index=False)
print("\nSaved: final_results.csv")

# Summary for paper
print("\n" + "="*70)
print("SUMMARY FOR PAPER")
print("="*70)
print(f"F1 Macro: {our_macro_f1:.4f}")
print(f"F1 Weighted: {our_weighted_f1:.4f}")
print(f"Runtime: {avg_runtime:.4f} s/sample")
print(f"GFLOPS: {gflops:.2f}")
print(f"Submission Score: {our_score:.4f}")